In [ ]:
import os
import json
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'figure.dpi':150,'reso':'xx-hi',
    'font.size':8,'label.size':8,'tick.labelsize':8,
    'leftlabelsize':8,'toplabelsize':8,
    'leftlabel.weight':'normal','toplabel.weight':'normal',
    'tick.minor':False})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
PREDSDIR  = CONFIGS['filepaths']['predictions']
LATRANGE  = CONFIGS['domain']['latrange']
LONRANGE  = CONFIGS['domain']['lonrange']
SPLIT     = 'test'

MODELS_8 = ['pod_bl','base_bl','base_all','nonparam_all','gauss_all','sr_rh','sr_thermo','sr_thermo_flux']
LABELS_8 = {
    'pod_bl':        'POD',
    'base_bl':       'Baseline NN ($B_L$)',
    'base_all':      'Baseline NN (all vars)',
    'nonparam_all':  'Nonparametric Kernel NN',
    'gauss_all':     'Parametric Kernel NN',
    'sr_rh':         r'SR-RH: $a\,\widehat{RH}^3 + b$',
    'sr_thermo':     r'SR-Thermo',
    'sr_thermo_flux':r'SR-Thermo+Flux',
}
COLORS_8 = {
    'pod_bl':        'blood',
    'base_bl':       '#5BA7DA',
    'base_all':      '#5BA7DA',
    'nonparam_all':  '#F2C85E',
    'gauss_all':     '#D42028',
    'sr_rh':         '#1B2C61',
    'sr_thermo':     '#1B2C61',
    'sr_thermo_flux':'#1B2C61',
}

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds.tp.load()

results = {}
for name in MODELS_8:
    fpath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(fpath):
        print(f'Missing: {name}')
        continue
    with xr.open_dataset(fpath) as ds:
        pred = ds.tp.load()
    if 'seed' in pred.dims:
        pred = pred.mean('seed')
    if 'complexity' in pred.dims:
        pred = pred.isel(complexity=0)
    ytrue,ypred    = xr.align(truetp,pred,join='inner')
    results[name]  = (ytrue.squeeze(),ypred.squeeze())

print(f'Loaded {len(results)}/{len(MODELS_8)} models.')

In [ ]:
def spatial_r2(ytrue,ypred):
    ssres = ((ytrue-ypred)**2).sum('time',skipna=True)
    sstot = ((ytrue-ytrue.mean('time',skipna=True))**2).sum('time',skipna=True)
    return (1-ssres/sstot).squeeze()

def spatial_bias(ytrue,ypred):
    return (ypred-ytrue).mean('time').squeeze()

names   = [n for n in MODELS_8 if n in results]
n       = len(names)
ncols   = 4
nrows   = int(np.ceil(n/ncols))

fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',refwidth=2,share=False)
axs.format(
    coast=True,borders=False,
    latlim=LATRANGE,lonlim=LONRANGE,
    latlines=5,lonlines=5,grid=False)
axs_flat = np.atleast_1d(axs).ravel()
mappable = None

for ax,name in zip(axs_flat,names):
    ytrue,ypred = results[name]
    r2          = spatial_r2(ytrue,ypred)
    m = ax.pcolormesh(
        r2.lon.values,r2.lat.values,r2.values,
        cmap='ColdHot',cmap_kw={'left':0.5},
        vmin=0,vmax=1,levels=11,extend='min')
    if mappable is None:
        mappable = m
    ax.format(title=LABELS_8.get(name,name))

for ax in axs_flat[n:]:
    ax.set_visible(False)

fig.colorbar(mappable,loc='b',label='R$^2$ (temporal, per grid point)',ticks=0.1,length=0.8)
fig.format(suptitle=f'Spatial Prediction Skill — {SPLIT} set')
pplt.show()
# fig.save('../figs/fig_spatial_r2.jpg')

In [ ]:
fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',refwidth=2,share=False)
axs.format(
    coast=True,borders=False,
    latlim=LATRANGE,lonlim=LONRANGE,
    latlines=5,lonlines=5,grid=False)
axs_flat = np.atleast_1d(axs).ravel()
mappable = None

for ax,name in zip(axs_flat,names):
    ytrue,ypred = results[name]
    bias        = spatial_bias(ytrue,ypred)
    vmax        = float(np.nanpercentile(np.abs(bias.values),98))
    m = ax.pcolormesh(
        bias.lon.values,bias.lat.values,bias.values,
        cmap='ColdHot_r',vmin=-vmax,vmax=vmax,levels=21,extend='both')
    if mappable is None:
        mappable = m
    ax.format(title=LABELS_8.get(name,name))

for ax in axs_flat[n:]:
    ax.set_visible(False)

fig.colorbar(mappable,loc='b',label='Mean Bias (mm/day, predicted $-$ observed)',length=0.8)
fig.format(suptitle=f'Spatial Mean Bias — {SPLIT} set')
pplt.show()
# fig.save('../figs/fig_spatial_bias.jpg')